In [ ]:

import os
import sys

REPO_NAME = "food-classification-deep-learning"
REPO_URL = "https://github.com/niRmana11/food-classification-deep-learning.git"

# Detect if running inside Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("[INFO] Running in Google Colab environment.")
    
    # Check if the repository directory already exists
    if not os.path.exists(f"/content/{REPO_NAME}"):
        print(f"[INFO] Cloning repository from {REPO_URL}...")
        !git clone {REPO_URL}
        %cd /content/{REPO_NAME}
    else:
        print("[INFO] Repository already present. Pulling latest updates from main...")
        %cd /content/{REPO_NAME}
        !git pull origin main
    
    # Ensure project root is on Python's path so imports like `src.*` work reliably
    if f"/content/{REPO_NAME}" not in sys.path:
        sys.path.insert(0, f"/content/{REPO_NAME}")

    # Verify GPU availability
    import tensorflow as tf
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        print(f"[SUCCESS] GPU detected: {gpus[0].name}")
        !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
    else:
        print("[WARNING] No GPU detected! Please go to: Runtime -> Change runtime type -> T4 GPU")
else:
    print("[INFO] Running in local environment.")


In [ ]:
# Navigate to the repository
%cd /content/food-classification-deep-learning

# Pull the latest commits from main (data loader and augmentations)
!git pull origin main

# Verify GPU availability and driver
import tensorflow as tf
print("TensorFlow Version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"[SUCCESS] GPU Detected: {gpus[0].name}")
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
else:
    print("[WARNING] No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU")


In [ ]:
import time
import numpy as np
import tensorflow as tf
from src.preprocessing.data_loader import get_food101_datasets

# Test parameters
BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)

print(f"[INFO] Loading datasets with batch_size={BATCH_SIZE}, image_size={IMAGE_SIZE}...")
start_load = time.time()

# Load datasets using our shared pipeline
train_ds, val_ds, test_ds = get_food101_datasets(
    data_dir="data/raw/food-101",
    splits_dir="data/splits",
    model_type="resnet50",   
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE
)

print(f"[INFO] Dataset pipelines constructed in {time.time() - start_load:.2f}s")


In [ ]:
# Measure raw pipeline speed (how fast tf.data delivers 50 batches without model compute)
print("[INFO] Profiling tf.data pipeline throughput over 50 batches...")
start_pipe = time.time()
batch_count = 0

for images, labels in train_ds.take(50):
    batch_count += 1

elapsed_pipe = time.time() - start_pipe
images_processed = batch_count * BATCH_SIZE
throughput = images_processed / elapsed_pipe

print(f"[RESULT] Processed {images_processed} images in {elapsed_pipe:.2f}s")
print(f"[RESULT] Data pipeline throughput: {throughput:.1f} images/second")


In [ ]:
from tensorflow.keras import layers, models

# Construct a baseline ResNet50 classifier
base_model = tf.keras.applications.ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model.trainable = False  # Feature extraction baseline

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(101, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("[INFO] Model successfully compiled. Running 1-epoch benchmark...")
epoch_start = time.time()

# Train for 1 epoch on a 200-batch subset to safely benchmark time and memory
history = model.fit(
    train_ds.take(200),
    validation_data=val_ds.take(50),
    epochs=1
)

epoch_duration = time.time() - epoch_start
print(f"\n[BENCHMARK RESULT] 200 batches completed in: {epoch_duration:.2f}s")


In [ ]:
import subprocess

# Query NVIDIA driver for exact GPU memory consumption
res = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,nounits,noheader"]
)
mem_used, mem_total = [int(x.strip()) for x in res.decode("utf-8").split(",")]

print("GPU FEASIBILITY REPORT")
print(f"Hardware:              Google Colab T4 GPU (16GB VRAM)")
print(f"Selected Batch Size:   {BATCH_SIZE}")
print(f"Input Image Size:      {IMAGE_SIZE}")
print(f"Peak VRAM Used:        {mem_used:,} MB / {mem_total:,} MB ({mem_used/mem_total*100:.1f}%)")
print(f"Data Throughput:       {throughput:.1f} images/sec")
print(f"Estimated Full Epoch:  ~{(epoch_duration / 200 * (68175 / BATCH_SIZE)) / 60:.1f} minutes")
print(f"OOM Risk Status:       NO OUT-OF-MEMORY DETECTED (SAFE)")

